# Synthetic Evaluation

Evaluates the model on the held-out synthetic test set (`data/test.csv`).
Uses the centralized `pipeline.eval` module for consistent, and unified metrics.

## 0 — Setup

In [ ]:
import sys
sys.path.insert(0, "../..")

import pandas as pd
from openai import OpenAI

from pipeline.data.preprocessing import preprocess_text
from pipeline.data.xml_utils import convert_to_xml
from pipeline.eval import batch_inference, compute_metrics, print_report, to_dataframe

# ── config ──────────────────────────────────────────────────────────────────
EVAL_SYSTEM = "full"   # "full" = bf16, "quantized" = Q4_K_M GGUF
MODEL_FULL  = "../../SFT/Qwen3-0.6B-Base_SFT"
MODEL_QUANT = "../../Final_model/Qwen3-0.6B-Base_SFT-Q4_K_M.gguf"
MODEL_ID    = MODEL_FULL if EVAL_SYSTEM == "full" else MODEL_QUANT
VLLM_PORT   = 9995
MAX_WORKERS = 100 if EVAL_SYSTEM == "full" else 1

## 1 — Load & Preprocess Data

Start vLLM before running inference:
```bash
VLLM_USE_V1="0" CUDA_VISIBLE_DEVICES="0" python -m vllm.entrypoints.openai.api_server \
  --model ../../SFT/Qwen3-0.6B-Base_SFT --port 9995 \
  --tensor-parallel-size 1 --max-model-len 4096 --generation-config vllm --dtype bfloat16
```

In [ ]:
eval_df = pd.read_excel("LLM Eval.xlsx")
eval_df.rename(columns={"Revision": "default_response_1"}, inplace=True)

eval_df = preprocess_text(eval_df, input_col="default_response_0", output_col="default_response_1")

eval_df["GT-XML"] = eval_df["default_response_1"].apply(convert_to_xml)
for tag in ["<anonymized_report>", "</anonymized_report>", "<anonymized_report/>",
            "<raw_report>", "</raw_report>", "<raw_report/>"]:
    eval_df["default_response_0"] = eval_df["default_response_0"].str.replace(tag, "", regex=False).str.strip()

print(f"{len(eval_df)} samples loaded.")
eval_df.head(2)

## 2 — Run Inference

In [ ]:
client = OpenAI(api_key="EMPTY", base_url=f"http://localhost:{VLLM_PORT}/v1")

notes = eval_df["default_response_0"].tolist()
eval_df["LLM-XML"] = batch_inference(
    notes, client, MODEL_ID,
    temperature=0.0, max_tokens=2048, max_workers=MAX_WORKERS,
)
eval_df.head(2)

## 3 — Compute Metrics

In [ ]:
result = compute_metrics(
    predictions  = eval_df["LLM-XML"].tolist(),
    ground_truths= eval_df["GT-XML"].tolist(),
    source_notes = eval_df["default_response_0"].tolist(),
)
print_report(result, title="Synthetic Test Set")

In [ ]:
# DataFrame
df_metrics = to_dataframe(result)
df_metrics

## 4 — Visualise

In [ ]:
import os
os.makedirs("results", exist_ok=True)
import plotly.graph_objects as go
from plotly.subplots import make_subplots

entity_tags = sorted(result.per_entity.keys())
f1s         = [result.per_entity[t].f1        for t in entity_tags]
gt_counts   = [result.per_entity[t].gt_count  for t in entity_tags]
tp_counts   = [result.per_entity[t].tp        for t in entity_tags]
fn_counts   = [g - t for g, t in zip(gt_counts, tp_counts)]

fig = make_subplots(rows=1, cols=2, specs=[[{"type": "xy"}, {"type": "polar"}]])
fig.add_trace(go.Bar(name="TP",  x=entity_tags, y=tp_counts, marker_color="green"), row=1, col=1)
fig.add_trace(go.Bar(name="FN",  x=entity_tags, y=fn_counts, marker_color="red"),   row=1, col=1)
fig.add_trace(go.Scatterpolar(r=f1s, theta=entity_tags, fill="toself", name="F1"),  row=1, col=2)

fig.update_layout(
    barmode="stack",
    title=f"Synthetic eval — micro-F1={result.micro_f1:.4f}  macro-F1={result.macro_f1:.4f}  xml-parse={result.xml_parse_rate:.4f}",
    font=dict(size=14),
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    height=600, width=1200,
)
fig.show()
fig.write_html("results/phi_tag_performance_composite.html")
print("Saved phi_tag_performance_composite.html")

## 5 — Export Results

In [ ]:
import os
os.makedirs("results", exist_ok=True)
df_metrics.to_csv("results/synth_eval_metrics.csv", index=False)
eval_df[["default_response_0", "GT-XML", "LLM-XML"]].to_csv("results/synth_eval_predictions.csv", index=False)
print("Results saved.")